<a href="https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Given the same observable, anonymized content-performance signals, does a trained classifier produce a more precise "review this first" queue than a transparent baseline scoring rule — when both are evaluated on the same held-out set of clients neither has seen before?

The decision this supports: which pages in a large content library get a human reviewer's limited time first. Getting this wrong either wastes reviewer effort on healthy pages or lets genuinely declining pages go unnoticed. The output isn't meant to auto-publish anything — it ranks candidates for a person to check.

In [2]:
import os

print("Current working directory:", os.getcwd())
print()
print("Contents of cwd:")
print(os.listdir("."))

Current working directory: /content

Contents of cwd:
['.config', 'sample_data']


In [3]:
import subprocess
result = subprocess.run(["find", "/content", "-iname", "content_refresh_anonymized.csv"], capture_output=True, text=True)
print(result.stdout if result.stdout else "Not found under /content — repo may not be cloned into this Colab session yet.")

Not found under /content — repo may not be cloned into this Colab session yet.


In [4]:
!git clone https://github.com/sadineniManushree/flyrank--internship__ml.git
%cd flyrank--internship__ml
!ls data/raw/

Cloning into 'flyrank--internship__ml'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 175 (delta 78), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 1.91 MiB | 13.38 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/flyrank--internship__ml
content_refresh_anonymized.csv


In [5]:
import pandas as pd

RANDOM_STATE = 42
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
print(f"Rows loaded: {len(df):,}")
print(f"Columns: {list(df.columns)}")

forbidden = {"client_name", "url", "domain", "title", "keyword", "query"}
present = forbidden.intersection(set(df.columns))
assert not present, f"Found identifying columns that should not be here: {present}"
print("Public-safety check passed: no client names, URLs, domains, titles, or queries in columns.")

df.head()

Rows loaded: 30,000
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Public-safety check passed: no client names, URLs, domains, titles, or queries in columns.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [6]:
print(len(df))


30000


In [7]:
before = len(df)
df_filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df_filtered = df_filtered.drop_duplicates(subset="content_id")
after = len(df_filtered)
print(f"Rows before filtering: {before:,}")
print(f"Rows after filtering:  {after:,}")
print(f"Rows excluded:         {before - after:,}")

Rows before filtering: 30,000
Rows after filtering:  30,000
Rows excluded:         0


In [8]:
df_filtered["is_declining_label"] = (df_filtered["trend_direction"] == "down").astype(int)
base_rate = df_filtered["is_declining_label"].mean()
print(f"Declining-label rows: {df_filtered['is_declining_label'].sum():,}")
print(f"Declining-label rate (base rate): {base_rate:.3f}")

Declining-label rows: 16,262
Declining-label rate (base rate): 0.542


30,000 rows from the FlyRank ML Internship starter dataset (data/raw/content_refresh_anonymized.csv). No client names, URLs, domains, or queries present — verified in code. Kept rows with impressions_90d > 0 and content_age_days ≥ 90 (0 rows excluded — all 30,000 already qualified). Label: is_declining_label = 1 where trend_direction == "down" → 16,262 declining (54.2% base rate).

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Assumptions: trend_direction == "down" is a usable proxy for "worth reviewing," not a quality judgment; clients behave independently enough that a client-level split prevents leakage.
Features: 90-day impressions/clicks/sessions (log-transformed), avg. position, content age, days since update, word/char count, engagement rate, scroll rate, CTR, AI-traffic %. No label-derived or ID features used.
Label definition: is_declining_label = 1 where trend_direction == "down".
Baseline: transparent rule — visibility (40%) + freshness risk (30%) + position opportunity (25%) + depth gap (5%), all from percentile ranks, no learned weights.
Validation design: client-holdout split — 20% of clients held out entirely, so train and test never share a client.
Leakage checks: verified zero client overlap between train and test sets before scoring anything; confirmed no label-derived columns were used as features.

In [16]:
# ── Section 3a: Baseline scoring rule — exact formula from scripts/02_baseline_score.py ──
def percentile_rank(series):
    return pd.to_numeric(series, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)

def normalize(series):
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    lo, hi = values.min(), values.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - lo) / (hi - lo)

df_filtered["visibility_score"] = percentile_rank(np.log1p(df_filtered["impressions_90d"]))
df_filtered["freshness_risk_score"] = percentile_rank(df_filtered["days_since_last_update"])
df_filtered["position_opportunity_score"] = (
    (1 - normalize(df_filtered["avg_position"].clip(lower=1, upper=50)))
    * df_filtered["visibility_score"]
    * (df_filtered["avg_position"] > 0).astype(int)
)
df_filtered["depth_gap_score"] = (1 - percentile_rank(df_filtered["word_count"])) * df_filtered["visibility_score"]

df_filtered["baseline_score"] = (
    0.40 * df_filtered["visibility_score"]
    + 0.30 * df_filtered["freshness_risk_score"]
    + 0.25 * df_filtered["position_opportunity_score"]
    + 0.05 * df_filtered["depth_gap_score"]
).clip(0, 1)

print(df_filtered["baseline_score"].describe())

count    30000.000000
mean         0.448901
std          0.215599
min          0.007958
25%          0.281418
50%          0.442861
75%          0.623330
max          0.941189
Name: baseline_score, dtype: float64


In [12]:
# ── Section 3b: Client-holdout split (the key leakage safeguard) ──────────
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df_filtered, groups=df_filtered["client_id"]))

train_df = df_filtered.iloc[train_idx].copy()
test_df = df_filtered.iloc[test_idx].copy()

# Verify no client appears in both sets — the actual leakage check
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,}")
print(f"Unique clients — train: {train_df['client_id'].nunique():,} | test: {test_df['client_id'].nunique():,}")
print(f"Client overlap between train and test: {len(overlap)} (should be 0)")

Train rows: 23,837 | Test rows: 6,163
Unique clients — train: 25 | test: 7
Client overlap between train and test: 0 (should be 0)


In [17]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df_filtered, groups=df_filtered["client_id"]))

train_df = df_filtered.iloc[train_idx].copy()
test_df = df_filtered.iloc[test_idx].copy()

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,}")
print(f"Client overlap: {len(overlap)} (should be 0)")

Train rows: 23,837 | Test rows: 6,163
Client overlap: 0 (should be 0)


In [18]:
X_train = prep(train_df)
X_test = prep(test_df)
y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

results = {}
baseline_scores = test_df["baseline_score"].values
results["baseline_rules"] = {
    "roc_auc": roc_auc_score(y_test, baseline_scores),
    "avg_precision": average_precision_score(y_test, baseline_scores),
    "precision_at_50": precision_at_k(y_test, baseline_scores, 50),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)
    results[name] = {
        "roc_auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
        "precision_at_50": precision_at_k(y_test, proba, 50),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
    }

results_df = pd.DataFrame(results).T
print(results_df.round(3))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                     roc_auc  avg_precision  precision_at_50  recall     f1
baseline_rules         0.498          0.482             0.32     NaN    NaN
logistic_regression    0.572          0.573             0.78   0.575  0.569
decision_tree          0.589          0.574             0.50   0.586  0.579
random_forest          0.612          0.596             0.64   0.714  0.636


In [19]:
print(df_filtered["baseline_score"].describe())
print(df_filtered["baseline_score"].nunique())
print(test_df["baseline_score"].corr(test_df["is_declining_label"]))

count    30000.000000
mean         0.448901
std          0.215599
min          0.007958
25%          0.281418
50%          0.442861
75%          0.623330
max          0.941189
Name: baseline_score, dtype: float64
29679
-0.02852802055464117


In [20]:
"logistic_regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)

SyntaxError: illegal target for annotation (1935444407.py, line 1)

In [21]:
print(df_filtered.index.is_unique)
print(df_filtered.index.equals(pd.RangeIndex(len(df_filtered))))

True
True


In [24]:
import inspect
# just to locate things — if this doesn't work in your notebook, ignore it and just scroll manually
print([name for name in dir() if 'baseline' in name.lower() or 'declin' in name.lower()])

['baseline_scores']


In [26]:
df["is_declining_label"] = (df["revenue_change_pct"] < -0.15).astype(int)

KeyError: 'revenue_change_pct'

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [14]:
# ── Section 4a: Prepare features ───────────────────────────────────────────
feature_cols = [
    "impressions_90d", "clicks_90d", "sessions_90d",
    "avg_position", "content_age_days", "days_since_last_update",
    "word_count", "char_count", "engagement_rate", "scroll_rate",
    "ctr", "ai_traffic_pct"
]

def prep(d):
    X = d[feature_cols].copy()
    # log-transform the count-based columns to reduce skew, matching the report's approach
    for col in ["impressions_90d", "clicks_90d", "sessions_90d"]:
        X[f"log_{col}"] = np.log1p(X[col])
    X = X.fillna(X.median(numeric_only=True))
    return X

X_train = prep(train_df)
X_test = prep(test_df)
y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

print(f"Feature matrix — train: {X_train.shape}, test: {X_test.shape}")

Feature matrix — train: (23837, 15), test: (6163, 15)


In [15]:
# ── Section 4b: Train the three models + score the baseline on test ───────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return y_true.iloc[order].mean()

results = {}

# Baseline (already computed as baseline_score, just score on test)
baseline_scores = test_df["baseline_score"].values
results["baseline_rules"] = {
    "roc_auc": roc_auc_score(y_test, baseline_scores),
    "avg_precision": average_precision_score(y_test, baseline_scores),
    "precision_at_50": precision_at_k(y_test, baseline_scores, 50),
}

models = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "decision_tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)
    results[name] = {
        "roc_auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
        "precision_at_50": precision_at_k(y_test, proba, 50),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
    }

import pandas as pd
results_df = pd.DataFrame(results).T
print(results_df.round(3))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                     roc_auc  avg_precision  precision_at_50  recall     f1
baseline_rules         0.432          0.447             0.26     NaN    NaN
logistic_regression    0.572          0.573             0.78   0.575  0.569
decision_tree          0.589          0.574             0.50   0.586  0.579
random_forest          0.612          0.596             0.64   0.714  0.636


## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
